In [64]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
import pandas as pd

# prep
data = pd.read_csv('polyx0data.csv')
X = data['Coefficients']
y = data['Starting Point']
X = X.apply(eval)

In [65]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train, X_test = torch.tensor(X_train.tolist(), requires_grad=True), torch.tensor(y_train.tolist(), requires_grad=True)
y_train, y_test = torch.unsqueeze(torch.tensor(y_train.tolist()), dim=1), torch.unsqueeze(torch.tensor(y_test.tolist()), dim=1)

In [66]:
import torch
import torch.nn as nn
import torch.optim as optim

class StartingPointNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.dense1 = nn.Linear(4, 16)
        self.dense2 = nn.Linear(16, 4)
        self.dense3 = nn.Linear(4, 1)

    def forward(self, x):
        x = self.dense1(x)
        x = torch.relu(x)
        x = self.dense2(x)
        x = torch.relu(x)
        x = self.dense3(x)
        return x

# define
model = StartingPointNN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def newton_solver(coeffs, x0, max_iter=10):
    def f(x):
        return sum(coef * x**i for i, coef in enumerate(coeffs))
    x = x0
    for _ in range(max_iter):
        fx = f(x)
        fx.backward()  # <- is always None
        x = x - fx / fx.grad
    return x

# loss function
def compute_loss(model, X, y):
    x0s = model(X)
    roots_pred = torch.stack([newton_solver(c, x0) for c, x0 in zip(X, x0s)])
    roots_label = torch.stack([newton_solver(c, y0) for c, y0 in zip(X, y)])
    loss = torch.mean((roots_pred - roots_label) ** 2)
    return loss

In [67]:
epochs = 90
for epoch in range(epochs):
	model.train()
	optimizer.zero_grad()
	y_pred = model(X_train)
	loss = compute_loss(model, X_train, y_train)
	loss.backward()
	optimizer.step()

	print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

/tmp/ipykernel_23713/1600700019.py:32: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  x = x - fx / fx.grad


TypeError: unsupported operand type(s) for /: 'Tensor' and 'NoneType'

In [9]:
'''# evaluate
y_pred = model.apply({'params': params}, X_test)
roots_pred = jax.vmap(newton_solver)(X_test, y_pred)
roots_test = jax.vmap(newton_solver)(X_test, y_test)
accuracy = jnp.mean(jnp.isclose(roots_pred, roots_test, atol=1e-2))
for r1, r2 in zip(roots_pred, roots_test):
	print(f'{r1.item(), r2.item()}')
print(f"\nTest Accuracy: {accuracy * 100:.2f}%")
'''

'# evaluate\ny_pred = model.apply({\'params\': params}, X_test)\nroots_pred = jax.vmap(newton_solver)(X_test, y_pred)\nroots_test = jax.vmap(newton_solver)(X_test, y_test)\naccuracy = jnp.mean(jnp.isclose(roots_pred, roots_test, atol=1e-2))\nfor r1, r2 in zip(roots_pred, roots_test):\n\tprint(f\'{r1.item(), r2.item()}\')\nprint(f"\nTest Accuracy: {accuracy * 100:.2f}%")\n'